In [2]:
import numpy as np
import pandas as pd


In [4]:
data = pd.read_csv("abalone.data", header=None)
data.columns = [
    "Sex", "Length", "Diameter", "Height",
    "Whole_weight", "Shucked_weight",
    "Viscera_weight", "Shell_weight", "Rings"
]

print("Number of rows:", len(data))
print("Column names:", data.columns)
print(data.head())


Number of rows: 4177
Column names: Index(['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight',
       'Viscera_weight', 'Shell_weight', 'Rings'],
      dtype='object')
  Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  Viscera_weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell_weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  


Physical measurements of abalone like length, diameter, weight are inputs

Rings (used to estimate age) is output

Age is a continuous quantity, not a category

In [5]:
y = data["Rings"] + 1.5


In [6]:
X = data[["Length", "Diameter", "Shell_weight"]].values


Feature 1: Length
Represents overall size of abalone

Feature 2: Diameter
Gives idea of body volume

Feature 3: Shell_weight
Related to maturity and age

In [7]:
split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(3341, 3) (3341,)
(836, 3) (836,)


In [8]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std


Normalization is needed because features have different scales
Without normalization, gradients become unstable
and learning becomes difficult


In [9]:
def forward(X, w, b):
    y_hat = X.dot(w) + b
    print("X shape:", X.shape)
    print("w shape:", w.shape)
    print("b shape:", type(b))
    print("y_hat shape:", y_hat.shape)
    return y_hat


Parameters are weights and bias

Number of parameters = 3 weights + 1 bias = 4

In [10]:
def mse(y, y_hat):
    loss = np.mean((y_hat - y) ** 2)
    return loss


Square to remove sign and penalize large errors

Large prediction errors are very costly under MSE


Gradient tells direction and rate at which loss increases

Moving opposite to gradient moves towards minimum loss

In [11]:
def grad_w(X, y, y_hat):
    N = len(y)
    return (2 / N) * X.T.dot(y_hat - y)

def grad_b(y, y_hat):
    N = len(y)
    return (2 / N) * np.sum(y_hat - y)


Meaning of large gradient:
Model is making big mistakes

Effect of too-large learning rate:
Training becomes unstable or diverges

In [12]:
np.random.seed(0)

w = np.random.randn(3)
b = 0.0

lr = 0.01
epochs = 1000

# Initial expectation:
# I expect loss to decrease slowly because dataset is noisy

for epoch in range(epochs):
    y_hat = X_train.dot(w) + b
    loss = mse(y_train, y_hat)

    dw = grad_w(X_train, y_train, y_hat)
    db = grad_b(y_train, y_hat)

    w = w - lr * dw
    b = b - lr * db

    if epoch % 100 == 0:
        print(epoch, loss)

# Revised expectation after training:
# Loss decreases steadily but not perfectly smooth


0 141.4281219527753
100 9.300117007311776
200 6.875342548927037
300 6.758794201102287
400 6.710388361562695
500 6.679645813483436
600 6.659100243367567
700 6.644712536492409
800 6.634088142983723
900 6.625801367063234


In [13]:
y_test_hat = X_test.dot(w) + b

test_mse = mse(y_test, y_test_hat)
test_mae = np.mean(np.abs(y_test - y_test_hat))

print("Test MSE:", test_mse)
print("Test MAE:", test_mae)


Test MSE: 5.134069696934935
Test MAE: 1.7290038032310746


In [14]:
for i in range(5):
    print(
        "True age:", y_test.iloc[i],
        "Predicted age:", y_test_hat[i],
        "Absolute error:", abs(y_test.iloc[i] - y_test_hat[i])
    )


True age: 13.5 Predicted age: 10.731391819688053 Absolute error: 2.768608180311947
True age: 15.5 Predicted age: 9.649447312830748 Absolute error: 5.850552687169252
True age: 14.5 Predicted age: 9.982987891212147 Absolute error: 4.517012108787853
True age: 14.5 Predicted age: 11.142525770249812 Absolute error: 3.3574742297501885
True age: 13.5 Predicted age: 11.395207644643097 Absolute error: 2.104792355356903


Systematic errors:
Older abalones tend to have larger prediction errors.


Observed bias:
The model shows a negative bias,
as it usually predicts lower age than the true value.

